# 99 Compare results

Collects every finished `results/<METHOD>/seed_<seed>/summary.json` (whatever has been run so far, on any machine: copy friends' `results/<METHOD>/` folders into `results/` first).

1. Per-method table: mean ± 95% CI of every KPI ratio vs RBC (lower is better) and the ranking score
2. Stage A ranking of the new methods M1–M6 and the top 3
3. Paired differences vs Static-Matched (B3) on shared seeds
4. Plots: cost vs carbon ratio, and KPI ratios per method

In [ ]:
%run ./00_core.ipynb
from scipy import stats
import matplotlib.pyplot as plt

## 0. No-battery reference (B5)

Batteries never used (action 0 every hour). CityLearn's own KPIs are normalised against this "no control" case, and in seed-42 results it beat the RBC, so every method is also scored against it below. Deterministic, about 1 minute, cached in `results/B5_no_battery/`.

In [ ]:
LABELS = {m: cfg['label'] for m, cfg in METHODS.items()}
LABELS['B5_no_battery'] = 'No battery (do nothing)'

def run_no_battery():
    """Batteries never charged or discharged; returns (and caches) its summary."""
    folder = RESULTS_DIR / 'B5_no_battery' / 'seed_0'
    if (folder / 'summary.json').exists():
        with open(folder / 'summary.json') as f:
            return json.load(f)
    folder.mkdir(parents=True, exist_ok=True)
    rbc = load_rbc()
    env = CityLearnEnv(schema=DATASET_NAME, central_agent=False)
    env.reset()
    n = len(env.buildings)
    tracker = new_tracker(n)
    rows = []
    while True:
        _, _, done, _ = env.step([[0.0]] * n)
        rows.append(trace_row(env, env.time_step, step_terms(env, tracker), [0.0] * n))
        if done:
            break
    trace = pd.DataFrame(rows)
    kpis = compute_kpis(trace, env.buildings)
    with open(RESULTS_DIR / RBC_FOLDER / 'summary.json') as f:
        rbc_kpis = json.load(f)['kpis']
    summary = {'method': 'B5_no_battery', 'label': LABELS['B5_no_battery'], 'seed': 0, 'kpis': kpis,
               'ratios_vs_rbc': kpi_ratios(kpis, rbc['trace'], rbc_kpis)}
    trace.to_csv(folder / 'eval_trace.csv', index=False)
    with open(folder / 'summary.json', 'w') as f:
        json.dump(summary, f, indent=2)
    return summary

no_battery = run_no_battery()
{k: round(v, 4) for k, v in no_battery['ratios_vs_rbc'].items()}

## Collect all finished runs

Adds `score_vs_nb` (the same score, but every KPI divided by the no-battery value instead of RBC) and a collapse check: `soc_std` is the battery state-of-charge spread over the year, averaged over buildings. Below 0.05 means the policy hardly uses the battery (`battery_idle`).

In [ ]:
def soc_spread(method, seed):
    trace = pd.read_csv(RESULTS_DIR / method / f'seed_{seed}' / 'eval_trace.csv')
    return float(trace[[c for c in trace.columns if c.startswith('soc_b')]].std().mean())

rows = []
for method in list(METHODS) + ['B5_no_battery']:
    for path in sorted((RESULTS_DIR / method).glob('seed_*/summary.json')):
        with open(path) as f:
            s = json.load(f)
        row = summary_row(s)
        row['score_vs_nb'] = float(np.mean([s['kpis'][k] / no_battery['kpis'][k] for k in SCORE_KPIS]))
        row['soc_std'] = soc_spread(method, s['seed'])
        rows.append(row)
all_runs = pd.DataFrame(rows)
all_runs['battery_idle'] = (all_runs['soc_std'] < 0.05) & (all_runs['method'] != 'B5_no_battery')
print(f"{len(all_runs)} finished runs")
all_runs[['method', 'seed', 'score', 'score_vs_nb', 'cost_ratio', 'carbon_ratio', 'soc_std', 'battery_idle']].round(4)

## 1. Mean ± 95% CI per method

In [ ]:
KPI_COLS = ['score', 'score_vs_nb'] + [f'{k}_ratio' for k in SCORE_KPIS] + ['carbon_per_kwh_imported_ratio', 'battery_cycles_ratio',
                                                          'electricity_consumption_ratio']

def mean_ci(values):
    """Mean and 95% CI half-width (t distribution); half-width is NaN with a single seed."""
    values = np.asarray(values, dtype=float)
    if len(values) < 2:
        return values.mean(), float('nan')
    return values.mean(), stats.t.ppf(0.975, len(values) - 1) * values.std(ddof=1) / np.sqrt(len(values))

def method_table(df):
    out = []
    for method, g in df.groupby('method', sort=False):
        row = {'method': method, 'label': LABELS[method], 'n_seeds': len(g), 'idle_seeds': int(g['battery_idle'].sum())}
        for c in KPI_COLS:
            m, h = mean_ci(g[c])
            row[c] = f"{m:.4f} ± {h:.4f}" if not np.isnan(h) else f"{m:.4f}"
        out.append(row)
    return pd.DataFrame(out).set_index('method')

table = method_table(all_runs)
table.to_csv(RESULTS_DIR / 'comparison_table.csv')
table

## 2. Stage A ranking of the new methods

Ranked by mean score over the seeds each method has (lower = better on average across cost, carbon, ramping, daily peak, annual peak and 1 − load factor). Compare only methods with the same seeds. `idle_seeds` > 0 flags seeds where the policy hardly used the battery.

In [ ]:
new = all_runs[all_runs['method'].str.startswith('M')]
ranking = new.groupby('method').agg(n_seeds=('seed', 'count'), score=('score', 'mean'), score_std=('score', 'std'),
                                   score_vs_nb=('score_vs_nb', 'mean'), cost=('cost_ratio', 'mean'),
                                   carbon=('carbon_ratio', 'mean'), idle_seeds=('battery_idle', 'sum')).sort_values('score')
print('Top 3:', list(ranking.index[:3]))
ranking.round(4)

## 3. Paired differences vs Static-Matched (B3)

For each method, the per-seed difference (method − B3) on seeds both have. Negative = better than B3. The CI excludes 0 only if the difference is consistent across seeds.

In [ ]:
base = all_runs[all_runs['method'] == 'B3_static_matched'].set_index('seed')
paired = []
for method, g in all_runs.groupby('method', sort=False):
    if method == 'B3_static_matched' or base.empty:
        continue
    g = g.set_index('seed')
    common = g.index.intersection(base.index)
    if len(common) == 0:
        continue
    row = {'method': method, 'seeds': list(common)}
    for c in ['score', 'cost_ratio', 'carbon_ratio', 'ramping_ratio', 'daily_peak_ratio']:
        m, h = mean_ci(g.loc[common, c] - base.loc[common, c])
        row[c] = f"{m:+.4f} ± {h:.4f}" if not np.isnan(h) else f"{m:+.4f}"
    paired.append(row)
pd.DataFrame(paired).set_index('method') if paired else 'No B3 runs yet'

## 4. Plots

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for method, g in all_runs.groupby('method', sort=False):
    _, hx = mean_ci(g['cost_ratio']); _, hy = mean_ci(g['carbon_ratio'])
    ax.errorbar(g['cost_ratio'].mean(), g['carbon_ratio'].mean(), xerr=0 if np.isnan(hx) else hx,
                yerr=0 if np.isnan(hy) else hy, fmt='o', capsize=4, label=LABELS[method])
ax.axvline(1.0, color='gray', ls='--', lw=0.8); ax.axhline(1.0, color='gray', ls='--', lw=0.8)
ax.set_xlabel('Cost ratio vs RBC (lower is better)'); ax.set_ylabel('Carbon ratio vs RBC (lower is better)')
ax.set_title('Cost vs carbon (mean ± 95% CI over seeds)'); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(PLOTS_DIR / 'cost_vs_carbon.png', dpi=200); plt.show()

In [ ]:
means = all_runs.groupby('method', sort=False)[[f'{k}_ratio' for k in SCORE_KPIS]].mean()
means.index = [LABELS[m] for m in means.index]
# matplotlib directly: pandas' .plot needs pkg_resources, which this venv does not have
fig, ax = plt.subplots(figsize=(12, 5))
width = 0.85 / len(means)
for i, (label, vals) in enumerate(means.iterrows()):
    ax.bar(np.arange(len(SCORE_KPIS)) + i * width, vals.values, width, label=label)
ax.axhline(1.0, color='black', lw=0.8); ax.set_ylabel('Ratio vs RBC (lower is better)')
ax.set_xticks(np.arange(len(SCORE_KPIS)) + 0.85 / 2 - width / 2)
ax.set_xticklabels([k.replace('_', ' ') for k in SCORE_KPIS]); ax.legend(fontsize=7, ncol=3)
ax.set_ylim(min(0.8, means.values.min() - 0.02), max(1.2, means.values.max() + 0.02))
plt.tight_layout(); plt.savefig(PLOTS_DIR / 'kpi_ratios.png', dpi=200); plt.show()